# Load data

In [12]:
# Train (fit) trên train.csv
# Tune / chọn hyperparam dựa trên dev.csv (validate)
# Chấm điểm 1 lần trên test.csv
# Dữ liệu đã preprocessing sẵn, cột: free_text, label_id
import os
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.model_selection import PredefinedSplit
from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score,
    classification_report, confusion_matrix
)
from scipy.stats import loguniform


# ===================== CONFIG =====================
TRAIN_PATH = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/train.csv"
DEV_PATH   = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/dev.csv"
TEST_PATH  = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/test.csv"

TEXT_COL  = "free_text"
LABEL_COL = "label_id"



# ===================== LOAD =====================
def load_xy(path):
    df = pd.read_csv(path).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
    df[TEXT_COL]  = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df[TEXT_COL].tolist(), df[LABEL_COL].tolist()

X_train, y_train = load_xy(TRAIN_PATH)
X_dev,   y_dev   = load_xy(DEV_PATH)
X_test,  y_test  = load_xy(TEST_PATH)

# (Optional) Kiểm tra “leak” trùng text giữa các split (cảnh báo thôi)
def overlap_count(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb)

print("Overlap train-dev:", overlap_count(X_train, X_dev))
print("Overlap train-test:", overlap_count(X_train, X_test))
print("Overlap dev-test:", overlap_count(X_dev, X_test))



# ===================== LOAD =====================
def load_xy(path):
    df = pd.read_csv(path).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
    df[TEXT_COL]  = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df[TEXT_COL].tolist(), df[LABEL_COL].tolist()


Overlap train-dev: 381
Overlap train-test: 897
Overlap dev-test: 134


# Train model

In [13]:
SCORING = "f1_macro"   # phù hợp multi-class + lệch nhãn
SAVE_MODEL_PATH = "./models/final_best_logreg_tfidf.joblib"
SAVE_INFO_PATH  = "./models/final_best_logreg_tfidf_info.json"
RANDOM_STATE = 42
N_ITER_WORD = 60   # giảm nếu muốn nhanh hơn
N_ITER_CHAR = 60

C_DIST = loguniform(1e-2, 1e1)

# ===================== PIPELINE TEMPLATE =====================
def make_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=5000, n_jobs=-1))
    ])

def eval_and_print(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("f1_macro:", f1_score(y_true, y_pred, average="macro"))
    print("acc     :", accuracy_score(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nReport:\n", classification_report(y_true, y_pred, digits=4))

# ===================== PredefinedSplit: train=-1, dev=0 =====================
X_train_dev = X_train + X_dev
y_train_dev = y_train + y_dev
test_fold = [-1] * len(X_train) + [0] * len(X_dev)
ps = PredefinedSplit(test_fold=test_fold)

# ===================== PARAM DISTRIBUTIONS =====================
SOLVERS_6 = ["liblinear", "saga", "lbfgs", "sag"]

# Chạy 2 search riêng để tránh combo “word params” lẫn “char params”
param_dist_word = {
    "tfidf__analyzer": ["word"],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "clf__solver": SOLVERS_6,
    "clf__C": C_DIST,
    "clf__penalty": ["l2"],
    "clf__class_weight": ["balanced"],
}

param_dist_char = {
    "tfidf__analyzer": ["char_wb"],
    "tfidf__ngram_range": [(3, 5), (4, 6)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "clf__solver": SOLVERS_6,
    "clf__C": C_DIST,
    "clf__penalty": ["l2"],
    "clf__class_weight": ["balanced"],
}
import numpy as np

# ===================== RANDOMIZED SEARCH (tune theo DEV) =====================
def run_random_search(param_dist, n_iter, tag):
    rs = RandomizedSearchCV(
        estimator=make_pipeline(),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=SCORING,
        cv=ps,                 # <- chỉ validate trên dev
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE,
        refit=False,           # <- tránh refit trên train+dev (để không “dính” dev lúc tune)
        error_score=np.nan      # <- combo lỗi (nếu có) sẽ bị bỏ qua thay vì crash
    )
    rs.fit(X_train_dev, y_train_dev)
    print(f"\n[{tag}] best_dev_score({SCORING}) = {rs.best_score_}")
    print(f"[{tag}] best_params = {rs.best_params_}")
    return rs.best_score_, rs.best_params_

word_score, word_params = run_random_search(param_dist_word, N_ITER_WORD, "WORD")
char_score, char_params = run_random_search(param_dist_char, N_ITER_CHAR, "CHAR")

if char_score >= word_score:
    best_params = char_params
    best_dev_score = char_score
    best_tag = "CHAR"
else:
    best_params = word_params
    best_dev_score = word_score
    best_tag = "WORD"

print("\n===== BEST (by DEV via RandomizedSearchCV) =====")
print("best_tag:", best_tag)
print("best_dev_score:", best_dev_score)
print("best_params:", best_params)

# ===================== Fit on TRAIN -> Report DEV =====================
best_model_train = make_pipeline()
best_model_train.set_params(**best_params)
best_model_train.fit(X_train, y_train)

dev_pred = best_model_train.predict(X_dev)
eval_and_print("DEV (train-only fit)", y_dev, dev_pred)

# ===================== FINAL: Fit TRAIN only -> TEST =====================
final_model = make_pipeline()
final_model.set_params(**best_params)

# Fit CHỈ trên TRAIN (không dùng DEV)
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)
eval_and_print("TEST (train-only final)", y_test, test_pred)


# ===================== SAVE =====================
Path(os.path.dirname(SAVE_MODEL_PATH)).mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, SAVE_MODEL_PATH)

info = {
    "best_tag": best_tag,
    "best_params": best_params,
    "best_dev_score": float(best_dev_score),
    "test_f1_macro": float(f1_score(y_test, test_pred, average="macro")),
    "test_acc": float(accuracy_score(y_test, test_pred)),
}
with open(SAVE_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print(f"\nSaved model: {SAVE_MODEL_PATH}")
print(f"Saved info : {SAVE_INFO_PATH}")

Fitting 1 folds for each of 60 candidates, totalling 60 fits


/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a o

[CV] END clf__C=0.010500232504231355, clf__class_weight=balanced, clf__penalty=l2, clf__solver=liblinear, tfidf__analyzer=word, tfidf__max_df=0.9, tfidf__min_df=5, tfidf__ngram_range=(1, 1); total time=   0.3s


/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a o

[CV] END clf__C=0.13292918943162169, clf__class_weight=balanced, clf__penalty=l2, clf__solver=liblinear, tfidf__analyzer=word, tfidf__max_df=1.0, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time=   0.6s
[CV] END clf__C=0.08200518402245831, clf__class_weight=balanced, clf__penalty=l2, clf__solver=liblinear, tfidf__analyzer=word, tfidf__max_df=0.95, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time=   0.6s
[CV] END clf__C=0.6251373574521749, clf__class_weight=balanced, clf__penalty=l2, clf__solver=lbfgs, tfidf__analyzer=word, tfidf__max_df=0.95, tfidf__min_df=5, tfidf__ngram_range=(1, 1); total time=   0.7s
[CV] END clf__C=0.023233503515390115, clf__class_weight=balanced, clf__penalty=l2, clf__solver=liblinear, tfidf__analyzer=word, tfidf__max_df=1.0, tfidf__min_df=5, tfidf__ngram_range=(1, 2); total time=   0.7s
[CV] END clf__C=0.010959604536925849, clf__class_weight=balanced, clf__penalty=l2, clf__solver=liblinear, tfidf__analyzer=word, tfidf__max_df=0.95, tfidf__min_df=2, t

...

[CV] END clf__C=2.6822739909301414, clf__class_weight=balanced, clf__penalty=l2, clf__solver=sag, tfidf__analyzer=char_wb, tfidf__max_df=1.0, tfidf__min_df=1, tfidf__ngram_range=(3, 5); total time= 3.7min

[CHAR] best_dev_score(f1_macro) = 0.6151282442279141
[CHAR] best_params = {'clf__C': np.float64(3.570962966813226), 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear', 'tfidf__analyzer': 'char_wb', 'tfidf__max_df': 0.95, 'tfidf__min_df': 2, 'tfidf__ngram_range': (3, 5)}

===== BEST (by DEV via RandomizedSearchCV) =====
best_tag: CHAR
best_dev_score: 0.6151282442279141
best_params: {'clf__C': np.float64(3.570962966813226), 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear', 'tfidf__analyzer': 'char_wb', 'tfidf__max_df': 0.95, 'tfidf__min_df': 2, 'tfidf__ngram_range': (3, 5)}


/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(



===== DEV (train-only fit) =====
f1_macro: 0.6151282442279141
acc     : 0.8514221556886228
Confusion matrix:
 [[2055   44   91]
 [  89   68   55]
 [  84   34  152]]

Report:
               precision    recall  f1-score   support

           0     0.9224    0.9384    0.9303      2190
           1     0.4658    0.3208    0.3799       212
           2     0.5101    0.5630    0.5352       270

    accuracy                         0.8514      2672
   macro avg     0.6327    0.6074    0.6151      2672
weighted avg     0.8445    0.8514    0.8467      2672



/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(



===== TEST (train-only final) =====
f1_macro: 0.6280408950849747
acc     : 0.8633233532934131
Confusion matrix:
 [[5227  131  190]
 [ 200  151   93]
 [ 228   71  389]]

Report:
               precision    recall  f1-score   support

           0     0.9243    0.9421    0.9331      5548
           1     0.4278    0.3401    0.3789       444
           2     0.5789    0.5654    0.5721       688

    accuracy                         0.8633      6680
   macro avg     0.6436    0.6159    0.6280      6680
weighted avg     0.8557    0.8633    0.8591      6680


Saved model: ./models/final_best_logreg_tfidf.joblib
Saved info : ./models/final_best_logreg_tfidf_info.json


# Test